# CPT Qwen3-4B **Instruct** (EKSPERIMEN) - gabungan 3 cache (fondasi + legal + kominfo), Base vs Instruct

> **STATUS: DISETUJUI TIM, SIAP JALAN.** Tim sudah memberi lampu hijau untuk eksperimen CPT-di-atas-Instruct ini.
> Jalankan via *Save Version -> Save & Run All (Commit)* di Kaggle (bukan sesi interaktif) supaya tetap
> berjalan walau browser/laptop ditutup. Estimasi ±10-12 jam (lihat sel 7b) - mendekati batas sesi 12 jam,
> kemungkinan butuh resume otomatis di sesi kedua kalau TimeGuard (`MAX_HOURS=11`) berhenti dulu.

**Revisi dari draf sebelumnya:** draf awal notebook ini salah mengartikan permintaan "pakai dataset Batch-1"
sebagai *mengganti* data domain (kominfo/legal) dengan data fondasi saja. Yang benar: **digabung** -
data fondasi Batch-1 **digabung** dengan data legal (round-2) dan kominfo (round-3) yang sudah ada, supaya
total volume data yang dilihat Instruct sebanding dengan total volume **kumulatif** yang pernah dilihat Base
di sepanjang 3 ronde-nya (bukan cuma volume ronde terakhir/kominfo saja yang kecil).

**Tujuan.** Menjawab secara empiris masukan Komdigi: *CPT lazimnya di atas base model, bukan instruct*.
Base mencapai hasil akhirnya lewat **3 ronde berurutan** (adapter dilanjutkan: fondasi -> +legal -> +kominfo).
Instruct **tidak bisa** melanjutkan adapter Base (bobot dasar beda), jadi sebagai pendekatan tersedekat:
**gabungkan ketiga cache jadi satu dataset** (single pass, LoRA baru), lalu bandingkan hasil akhirnya dengan
hasil akhir kumulatif Base round-3 (dicatat di `tim-A/cpt/kominfo/cpt-quen3-4b-id-kominfo-results.ipynb`:
kominfo 2,874->2,183, legal 2,894->2,118, news 5,760->5,048, wiki 9,280->8,319, socmed 18,545->17,283).

**Komposisi data gabungan (3 cache yang sudah ada di `kaggle_upload/`, BUKAN subsample baru):**

| Cache | Isi | Token |
|---|---|---|
| `cpt_indo_cache_10M` | fondasi: news/wiki/socmed (round-1) | 10.000.384 |
| `cpt_legal_cache_10M` | legal + replay umum (round-2) | 12.999.680 |
| `cpt_kominfo_cache` | kominfo + replay legal+umum (round-3) | 2.098.176 |
| **Total gabungan** | | **25.098.240 (~25,1 jt)** |

**Catatan jujur:** ketiga cache sudah di-*pack* & diacak per sumbernya sendiri-sendiri sebelum ini (tidak bisa
dipisah lagi per domain), jadi gabungan ini punya sedikit duplikasi replay (data umum muncul di 3 cache, legal
muncul di 2 cache). Bukan gabungan yang steril, tapi ini **data yang sama persis** yang pernah dilihat Base
secara kumulatif - jadi tetap pembanding yang wajar, bukan data baru/berbeda.

**Yang diukur:**
1. **Perplexity** kominfo/legal/news/wiki/socmed - dibandingkan dengan hasil akhir kumulatif Base round-3.
2. **Instruction-following sebelum vs sesudah CPT** (pakai chat template) - inilah yang dikhawatirkan Komdigi:
   apakah CPT (teks mentah, tanpa format instruksi) merusak kemampuan ikut-instruksi hasil SFT.

**Input Kaggle (Add Input, 3 dataset terpisah):** unggah ketiga folder lokal `kaggle_upload/cpt_indo_cache_10M/`,
`kaggle_upload/cpt_legal_cache_10M/`, `kaggle_upload/cpt_kominfo_cache/` sebagai 3 Kaggle Dataset privat.
Notebook mengenali ketiganya otomatis dari isi filenya (lihat sel konfigurasi/muat data). **TIDAK perlu**
adapter base: LoRA dipasang **baru** di atas bobot Instruct.

## Mengapa CPT biasanya di Base, bukan Instruct (ringkasan riset)

Masukan Komdigi benar sebagai *default*. Ringkasnya:

**Pipeline ortodoks:** `Base -> CPT (domain, teks mentah) -> SFT (instruksi) -> (opsional) RLHF/DPO`.
CPT = objektif *next-token* di teks mentah, **sama persis** dengan objektif pra-pelatihan base. Jadi CPT di atas
Base tidak "melawan" apa pun.

**Instruct = Base + SFT (+ sering RLHF/DPO).** Menjalankan CPT teks-mentah di atas Instruct berarti menarik
model kembali ke distribusi "melanjutkan teks", sehingga berisiko:
1. **Lupa instruction-following / format chat** (sebagian SFT ter-overwrite).
2. **Regresi alignment/safety.**
3. **Mismatch distribusi**: teks domain tidak punya struktur chat, sementara Instruct dioptimalkan untuk struktur itu.

**Kapan CPT-di-Instruct masih masuk akal / praktik "domain-adaptive post-training":**
- Kalau tujuannya **menyuntik pengetahuan domain tanpa mengulang SFT** (hemat tahap).
- Dengan **LoRA (bukan full-FT)** + **LR kecil** + **1 epoch** + **replay berat** + idealnya **sebagian data
  berformat instruksi/chat**, kerusakan instruction-following bisa ditekan.

**Mitigasi yang dipakai notebook ini:** LoRA only (bobot Instruct beku), LR 5e-5, 1 epoch, replay umum+legal
sudah *built-in* di ketiga cache yang digabung (lihat tabel komposisi di sel 0), dan **eval instruction-following
sebelum/sesudah** supaya efek ke SFT terukur, bukan ditebak. Keterbatasan jujur: data kami **teks mentah**,
belum ada data berformat chat untuk replay SFT - jadi eksperimen ini adalah *skenario terburuk yang terkontrol*
(kalau instruction-following tetap bertahan di sini, itu kabar baik; kalau turun, itu konfirmasi kuat untuk
kekhawatiran Komdigi). Catatan metodologi: Base mencapai komposisi data ini lewat **3 ronde berurutan**
(melanjutkan adapter antar ronde), sedangkan Instruct di notebook ini memakai **1 pass gabungan** (LoRA baru) -
bukan replikasi prosedur yang identik, tapi pendekatan tersedekat yang mungkin dilakukan untuk Instruct.

**Rekomendasi saya:** jalur aman tetap **CPT-di-Base lalu SFT**. Eksperimen Instruct ini cukup **satu A/B**
untuk menjawab Komdigi dengan data, dan **ditahan sampai tim setuju**.

## Versi Kaggle: cara pakai (baca dulu)
Sama seperti notebook Base, hanya berbeda model, 3 dataset digabung jadi satu training run, dan tambahan eval instruksi.

**Persiapan:**
1. Akun Kaggle terverifikasi (GPU + Internet).
2. *Add Input* -> unggah **3 Kaggle Dataset privat terpisah** dari folder lokal:
   - `kaggle_upload/cpt_indo_cache_10M/` (fondasi, ±42 MB)
   - `kaggle_upload/cpt_legal_cache_10M/` (legal, ±55 MB)
   - `kaggle_upload/cpt_kominfo_cache/` (kominfo, ±14 MB)
   Notebook mengenali ketiganya otomatis lewat nama file khasnya masing-masing (sel "Muat & gabungkan 3 cache") -
   urutan upload tidak masalah.
3. Panel kanan: *Accelerator* = **GPU T4 x2**, *Internet* = **On**.

**Menjalankan:** coba interaktif sampai sel **7b** (uji muat & estimasi). **Perhatian:** total data gabungan
±25,1 juta token diperkirakan butuh **±10-12 jam** - mendekati/melebihi batas sesi Kaggle 12 jam. Notebook
otomatis checkpoint & bisa **resume** (jalankan ulang semua sel di sesi baru, training lanjut dari checkpoint
terakhir) kalau perlu lanjut di sesi kedua. Cek kuota GPU mingguan (±30 jam) cukup untuk 2 sesi.

**Hasil:** tab *Output* -> `hasil_cpt_instruct.zip` (adapter + log + perplexity + hasil eval instruksi before/after).
Bandingkan `run_4b_instruct_merged.json` (notebook ini) dengan hasil akhir kumulatif Base round-3
(`tim-A/cpt/kominfo/cpt-quen3-4b-id-kominfo-results.ipynb`).

*Catatan jujur:* notebook ini belum pernah di-run (hanya logika path & ukuran file diuji). Kalau ada error
pip/load/deteksi folder, kirim pesannya.

## 0. Setup environment Kaggle
- `CUDA_VISIBLE_DEVICES=0`: Unsloth hanya memakai satu GPU.
- `pip install unsloth` sebelum `import torch`.
- Cek GPU: berhenti awal kalau Accelerator mati atau GPU P100 (compute capability < 7.0).

In [1]:
import os, subprocess, sys
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'unsloth'], capture_output=True, text=True)
print(r.stdout[-1500:] or 'pip selesai', r.stderr[-1500:] if r.returncode else '')
assert r.returncode == 0, 'pip install unsloth gagal (Internet = On? lihat pesan di atas)'

import torch, importlib.metadata as md_
assert torch.cuda.is_available(), 'GPU tidak aktif: Notebook options -> Accelerator -> GPU T4 x2'
cap = torch.cuda.get_device_capability(0)
print(torch.__version__, '|', torch.cuda.get_device_name(0), f'| compute capability {cap[0]}.{cap[1]} | VRAM {torch.cuda.get_device_properties(0).total_memory / 2**30:.1f} GB')
assert cap[0] >= 7, 'GPU terlalu lama untuk Unsloth (P100?) -> pilih GPU T4 x2'
for pkg in ['unsloth', 'transformers', 'peft', 'bitsandbytes']: print(pkg, md_.version(pkg))

pip selesai 
2.11.0+cu130 | Tesla T4 | compute capability 7.5 | VRAM 14.6 GB
unsloth 2026.11.3
transformers 5.17.0
peft 0.21.2
bitsandbytes 0.50.2


## 1. Konfigurasi
Beda inti dari notebook Base ada di baris bertanda **(INSTRUCT)**.

| Parameter | Nilai | Alasan |
|---|---|---|
| `MODEL_NAME` | `Qwen3-4B-Instruct-2507` **(INSTRUCT)** | Non-thinking instruct. Base pembanding = hasil akhir kumulatif round-3 (`Qwen3-4B-Base` + adapter berurutan) |
| `FRESH_LORA` | `True` **(INSTRUCT)** | Pasang LoRA **baru** di atas bobot Instruct. Adapter yang dilatih di atas Base **tidak** dipindah ke Instruct (bobot dasarnya beda) |
| `SOURCES` | `['kominfo','legal','news','wiki','socmed']` | Lima domain yang ada test set-nya di cache `cpt_kominfo_cache` |
| Data training | **gabungan 3 cache** (~25,1 jt token, lihat sel 0) | BUKAN subsample baru - memakai ulang persis cache fondasi+legal+kominfo yang sudah ada, supaya total volume sebanding dengan kumulatif Base |
| `MAX_SEQ_LEN` | 1024 | Sama dengan semua run sebelumnya |
| `LR` | 5e-5 | Konservatif; untuk Instruct makin penting agar SFT tidak "terguncang" |
| `BATCH x GRAD_ACC` | 1 x 16 | Sama |
| Epoch | 1 | Sama; mengulang data menaikkan risiko lupa SFT |

Output di folder terpisah `_cpt_instruct_output`.

In [2]:
import os, shutil, json, math, time
import unsloth   # harus diimpor sebelum transformers

from pathlib import Path
from collections import defaultdict
from tqdm.auto import tqdm
import numpy as np
INPUT = Path('/kaggle/input')

# --- (INSTRUCT) pilihan model ---
MODEL_NAME = 'unsloth/Qwen3-4B-Instruct-2507-unsloth-bnb-4bit'   # ganti kalau nama unsloth berbeda; cek: huggingface.co/unsloth
TAG        = '4b_instruct'
NEED_VRAM_GB = 3.4
FRESH_LORA = True                 # (INSTRUCT) True = LoRA baru di atas Instruct; JANGAN pakai adapter base
FORCE_TRY  = False
MAX_SEQ_LEN = 1024
SOURCES     = ['kominfo', 'legal', 'news', 'wiki', 'socmed']   # 5 domain -- cocok dengan test set di cache kominfo

LORA_R   = 16
LR       = 5e-5
BATCH, GRAD_ACC = 1, 16
EVAL_STEPS, SAVE_STEPS = 100, 25
SEED = 42

OUT_DIR = Path('/kaggle/working/_cpt_instruct_output')
OUT_DIR.mkdir(parents=True, exist_ok=True)
DATA_CACHE = OUT_DIR / f'data_merged_seq{MAX_SEQ_LEN}_seed{SEED}'   # gabungan 3 cache, bukan subsample bertarget token
DATA_CACHE.mkdir(parents=True, exist_ok=True)
CKPT_DIR    = OUT_DIR / f'checkpoints_{TAG}_merged'
ADAPTER_DIR = OUT_DIR / f'adapter_cpt_{TAG}_merged'
print(f'Model: {MODEL_NAME} | tag: {TAG} | FRESH_LORA={FRESH_LORA} | SOURCES={SOURCES}')

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: xformers is installed but cannot load with torch 2.11.0+cu130 (it was built for 2.10.0+cu128), so PyTorch attention is used instead.
To install a matching xformers, run `unsloth install-kernels xformers`, or `pip uninstall xformers` to hide this. UNSLOTH_ENABLE_LOGGING=1 shows the full error.
🦥 Unsloth Zoo will now patch everything to make training faster!
Model: unsloth/Qwen3-4B-Instruct-2507-unsloth-bnb-4bit | tag: 4b_instruct | FRESH_LORA=True | SOURCES=['kominfo', 'legal', 'news', 'wiki', 'socmed']


## 2. Muat & gabungkan 3 cache ter-packing (ganti tahap tokenizer-ratio + sampling + packing)
Karena ketiganya **sudah** ter-tokenisasi & ter-packing (hasil run Base sebelumnya), tidak perlu membaca
`train.jsonl` atau menghitung rasio tokenizer lagi. Notebook mengenali ketiga folder dari file khasnya:
- folder dengan `test_kominfo_ids.npy` -> cache **kominfo**
- folder dengan `test_legal_ids.npy` (tapi tanpa `test_kominfo_ids.npy`) -> cache **legal**
- sisanya -> cache **fondasi**

`train_blocks` dari ketiganya digabung (`concatenate`) lalu diacak ulang (`shuffle`) supaya blok antar-sumber
tercampur selama training, bukan berurutan per-cache. Val & test diambil dari cache **kominfo** karena itu
satu-satunya yang punya kelima domain sekaligus (dan angkanya cocok dengan baseline Base round-3 yang sudah dipublikasikan).

In [3]:
cache_dirs = sorted({p.parent for p in INPUT.rglob('train_blocks.npy')})
assert len(cache_dirs) == 3, (f'Perlu tepat 3 dataset ter-upload (fondasi, legal, kominfo) di Add Input, '
                              f'ditemukan {len(cache_dirs)}: {cache_dirs}')
INDO_DIR = LEGAL_DIR = KOMINFO_DIR = None
for d in cache_dirs:
    files = {f.name for f in d.iterdir()}
    if 'test_kominfo_ids.npy' in files: KOMINFO_DIR = d
    elif 'test_legal_ids.npy' in files: LEGAL_DIR = d
    else: INDO_DIR = d
assert INDO_DIR and LEGAL_DIR and KOMINFO_DIR, f'Gagal mengenali ketiga cache dari isinya: {cache_dirs}'
print('fondasi:', INDO_DIR, '\nlegal  :', LEGAL_DIR, '\nkominfo:', KOMINFO_DIR)

def to_blocks(ids, n_blocks=None):
    n = len(ids) // MAX_SEQ_LEN
    if n_blocks is not None:
        assert n >= n_blocks, f'token kurang: butuh {n_blocks} blok, ada {n}'
        n = n_blocks
    return ids[:n * MAX_SEQ_LEN].reshape(n, MAX_SEQ_LEN)

blocks_path = DATA_CACHE / 'train_blocks.npy'
if blocks_path.exists():
    train_blocks = np.load(blocks_path)
    print('Cache gabungan ditemukan -> muat ulang dilewati:', DATA_CACHE)
else:
    train_blocks = np.concatenate([np.load(d / 'train_blocks.npy') for d in (INDO_DIR, LEGAL_DIR, KOMINFO_DIR)])
    np.random.default_rng(SEED).shuffle(train_blocks)
    np.save(blocks_path, train_blocks)

val_ids = np.load(KOMINFO_DIR / 'val_ids.npy')
test_ids = {s: np.load(KOMINFO_DIR / f'test_{s}_ids.npy') for s in SOURCES}

print(f'TRAIN gabungan: {train_blocks.shape[0]:,} blok x {MAX_SEQ_LEN} = {train_blocks.size:,} token')
print(f'VAL  : {len(val_ids):,} token')
for s in SOURCES: print(f'TEST {s:7s}: {len(test_ids[s]):,} token')

fondasi: /kaggle/input/datasets/mikailachmad/cpt-instruct/cpt_indo_cache_10M 
legal  : /kaggle/input/datasets/mikailachmad/cpt-instruct/cpt_legal_cache_10M 
kominfo: /kaggle/input/datasets/mikailachmad/cpt-instruct/cpt_kominfo_cache
TRAIN gabungan: 24,510 blok x 1024 = 25,098,240 token
VAL  : 337,624 token
TEST kominfo: 147,072 token
TEST legal  : 250,000 token
TEST news   : 250,000 token
TEST wiki   : 248,070 token
TEST socmed : 177,539 token


## 5. Load model Instruct 4-bit
Dimuat 4-bit (QLoRA). Baseline dan model sesudah CPT diukur dengan model 4-bit yang sama, jadi selisih PPL
murni dari adapter. **(INSTRUCT)** model ini punya chat template - dipakai di eval instruksi (sel 6b & 10b).

In [4]:
from unsloth import FastLanguageModel, is_bfloat16_supported

free_b, total_b = torch.cuda.mem_get_info()
print(f'VRAM bebas: {free_b / 2**30:.2f} GB dari {total_b / 2**30:.2f} GB | butuh +-{NEED_VRAM_GB} GB')
if free_b / 2**30 < NEED_VRAM_GB and not FORCE_TRY:
    raise RuntimeError(f'VRAM bebas ({free_b / 2**30:.2f} GB) < kebutuhan ({NEED_VRAM_GB} GB). Tutup aplikasi GPU lain, atau FORCE_TRY=True.')

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME, max_seq_length=MAX_SEQ_LEN, dtype=None, load_in_4bit=True)
print(f'VRAM terpakai: {torch.cuda.memory_allocated() / 2**30:.2f} GB | chat_template ada? {tokenizer.chat_template is not None}')

VRAM bebas: 14.41 GB dari 14.56 GB | butuh +-3.4 GB
==((====))==  Unsloth 2026.11.3: Fast Qwen3 patching. Transformers: 5.17.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


/usr/local/lib/python3.13/dist-packages/huggingface_hub/constants.py:302: FutureWarning: The `HF_HUB_ENABLE_HF_TRANSFER` environment variable is deprecated as 'hf_transfer' is not used anymore. Please use `HF_XET_HIGH_PERFORMANCE` instead to enable high performance transfer with Xet. Visit https://huggingface.co/docs/huggingface_hub/package_reference/environment_variables#hfxethighperformance for more details.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/huggingface_hub/constants.py:302: FutureWarning: The `HF_HUB_ENABLE_HF_TRANSFER` environment variable is deprecated as 'hf_transfer' is not used anymore. Please use `HF_XET_HIGH_PERFORMANCE` instead to enable high performance transfer with Xet. Visit https://huggingface.co/docs/huggingface_hub/package_reference/environment_variables#hfxethighperformance for more details.
  warnings.warn(


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

VRAM terpakai: 3.31 GB | chat_template ada? True


## 6. Perplexity baseline (sebelum CPT)
**PPL** = exp(rata-rata loss per token); makin kecil makin baik. Diukur per sumber (kominfo/legal/news/wiki/socmed).
Disimpan ke `baseline_ppl_4b_instruct.json`. Angka ini untuk **konteks** dibandingkan dengan baseline Base
round-3 kumulatif (`kominfo 2,874, legal 2,894, news 5,760, wiki 9,280, socmed 18,545`) - walau bobot model
beda (Base vs Instruct) sehingga baseline PPL tidak harus identik, yang penting dibandingkan **before/after
di model yang sama** (sel 10).

In [5]:
@torch.no_grad()
def perplexity(model, ids, desc=''):
    model.eval()
    nll, n = 0.0, 0
    for i in tqdm(range(0, len(ids) - 1, MAX_SEQ_LEN), desc=desc, leave=False):
        x = torch.from_numpy(ids[i:i + MAX_SEQ_LEN].astype(np.int64))[None].cuda()
        if x.shape[1] < 2: continue
        loss = model(input_ids=x, labels=x).loss.float().item()
        nll += loss * (x.shape[1] - 1); n += x.shape[1] - 1
    return {'ppl': math.exp(nll / n), 'loss': nll / n, 'tokens': n}

base_path = DATA_CACHE / f'baseline_ppl_{TAG}.json'
if base_path.exists():
    baseline = json.loads(base_path.read_text())
else:
    baseline = {s: perplexity(model, test_ids[s], f'baseline {s}') for s in SOURCES if s in test_ids}
    base_path.write_text(json.dumps(baseline, indent=1))
for s in baseline: print(f'BASELINE {s:7s} PPL = {baseline[s]["ppl"]:7.3f}  (loss {baseline[s]["loss"]:.4f}, {baseline[s]["tokens"]:,} token)')

baseline kominfo:   0%|          | 0/144 [00:00<?, ?it/s]

baseline legal:   0%|          | 0/245 [00:00<?, ?it/s]

baseline news:   0%|          | 0/245 [00:00<?, ?it/s]

baseline wiki:   0%|          | 0/243 [00:00<?, ?it/s]

baseline socmed:   0%|          | 0/174 [00:00<?, ?it/s]

BASELINE kominfo PPL =   3.562  (loss 1.2703, 146,928 token)
BASELINE legal   PPL =   3.434  (loss 1.2338, 249,755 token)
BASELINE news    PPL =   7.357  (loss 1.9957, 249,755 token)
BASELINE wiki    PPL =  11.951  (loss 2.4808, 247,827 token)
BASELINE socmed  PPL =  23.365  (loss 3.1512, 177,365 token)


## 6b. (INSTRUCT) Instruction-following BASELINE - pakai chat template
Inilah pengukuran inti kekhawatiran Komdigi. **Sebelum** CPT, kita jalankan sekumpulan instruksi lewat
`apply_chat_template` dan simpan jawabannya. Nanti (sel 10b) diulang **sesudah** CPT dengan prompt sama,
lalu dibandingkan berdampingan untuk melihat apakah kemampuan ikut-instruksi menurun.

Sinyal otomatis ringan yang dicatat (pelengkap, bukan pengganti baca manual):
- **selesai_rapi**: model mengakhiri dengan token akhir chat (tidak "ngelantur" tanpa berhenti).
- **rasio_ulang**: proporsi 3-gram berulang (tinggi = degenerasi/mengulang).
- **panjang**: jumlah token jawaban.

In [6]:
INSTR_PROMPTS = [
    'Buatkan 3 poin Latar Belakang singkat untuk TOR kegiatan pelatihan literasi digital aparatur.',
    'Jelaskan dalam 2 kalimat apa itu RAB dalam pengadaan pemerintah.',
    'Sebutkan 3 contoh dasar hukum yang lazim dikutip dalam TOR Kementerian Komunikasi dan Digital.',
    'Ringkas kalimat berikut menjadi satu frasa: "Kegiatan ini bertujuan meningkatkan kapasitas aparatur dalam transformasi digital nasional."',
    "Terjemahkan ke bahasa Inggris: 'Kegiatan ini bertujuan meningkatkan kapasitas aparatur.'",
    'Apa ibu kota Provinsi Jawa Tengah? Jawab satu kata.',
    'Tuliskan balasan e-mail sopan untuk menolak undangan rapat karena bentrok jadwal.',
    'Format jawaban sebagai daftar bernomor 1-3: tiga manfaat transformasi digital bagi pelayanan publik.',
]

def ngram_repeat_ratio(ids, n=3):
    if len(ids) < n + 1: return 0.0
    grams = [tuple(ids[i:i+n]) for i in range(len(ids)-n+1)]
    return 1.0 - len(set(grams)) / len(grams)

@torch.no_grad()
def chat_generate(model, instr, max_new_tokens=200):
    msgs = [{'role': 'user', 'content': instr}]
    text = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    x = tokenizer(text, return_tensors='pt').to('cuda')
    torch.manual_seed(SEED)
    y = model.generate(**x, max_new_tokens=max_new_tokens, do_sample=False,
                       repetition_penalty=1.1, pad_token_id=tokenizer.eos_token_id)
    gen_ids = y[0][x['input_ids'].shape[1]:]
    selesai_rapi = bool((gen_ids == tokenizer.eos_token_id).any().item()) or (len(gen_ids) < max_new_tokens)
    out = tokenizer.decode(gen_ids, skip_special_tokens=True)
    return {'text': out, 'selesai_rapi': selesai_rapi,
            'rasio_ulang': round(ngram_repeat_ratio(gen_ids.tolist()), 3), 'panjang': int(len(gen_ids))}

FastLanguageModel.for_inference(model)
instr_before = []
for p in INSTR_PROMPTS:
    r = chat_generate(model, p); instr_before.append(r)
    print('INSTRUKSI:', p)
    print(f'  [sebelum CPT] selesai_rapi={r["selesai_rapi"]} rasio_ulang={r["rasio_ulang"]} panjang={r["panjang"]}')
    print('  ', r['text'][:400].replace('\n', ' '))
    print('-' * 100)
(OUT_DIR / 'instr_before.json').write_text(json.dumps(
    [{'prompt': p, **r} for p, r in zip(INSTR_PROMPTS, instr_before)], indent=1, ensure_ascii=False), encoding='utf-8')

Both `max_new_tokens` (=200) and `max_length`(=262144) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=200) and `max_length`(=262144) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


INSTRUKSI: Buatkan 3 poin Latar Belakang singkat untuk TOR kegiatan pelatihan literasi digital aparatur.
  [sebelum CPT] selesai_rapi=False rasio_ulang=0.015 panjang=200
   Tentu, berikut adalah 3 poin latar belakang singkat yang dapat digunakan dalam dokumen TOR (Tujuan dan Objek Rencana) kegiatan pelatihan literasi digital aparatur:  1. Dalam era digitalisasi, kemampuan aparatur pemerintah dalam mengelola teknologi informasi menjadi kunci utama dalam meningkatkan efisiensi, transparansi, dan kualitas layanan publik.  2. Banyak aparatur masih terbatas dalam pengguna
----------------------------------------------------------------------------------------------------


Both `max_new_tokens` (=200) and `max_length`(=262144) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


INSTRUKSI: Jelaskan dalam 2 kalimat apa itu RAB dalam pengadaan pemerintah.
  [sebelum CPT] selesai_rapi=True rasio_ulang=0.02 panjang=102
   RAB (Rencana Anggaran Bangunan) adalah dokumen yang menyajikan perhitungan biaya pembangunan suatu proyek secara rinci, termasuk harga material, tenaga kerja, dan biaya lainnya. Dalam pengadaan pemerintah, RAB digunakan sebagai dasar penilaian kebutuhan anggaran dan untuk menjamin bahwa proyek dapat dijalankan sesuai dengan anggaran yang tersedia.
----------------------------------------------------------------------------------------------------


Both `max_new_tokens` (=200) and `max_length`(=262144) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


INSTRUKSI: Sebutkan 3 contoh dasar hukum yang lazim dikutip dalam TOR Kementerian Komunikasi dan Digital.
  [sebelum CPT] selesai_rapi=False rasio_ulang=0.056 panjang=200
   Berikut adalah **3 contoh dasar hukum yang lazim dikutip dalam Tender of Request (TOR)** Kementerian Komunikasi dan Teknologi Informasi (Kominfo), terutama dalam konteks pengadaan barang, jasa, atau layanan yang berkaitan dengan komunikasi dan digital:  1. **Undang-Undang Nomor 2 Tahun 2009 tentang Pengelolaan Pemerintahan Secara Elektronik (UU PPE)**      → Dasar hukum ini mendukung pelaksanaan p
----------------------------------------------------------------------------------------------------


... (3 outputs left out)

INSTRUKSI: Terjemahkan ke bahasa Inggris: 'Kegiatan ini bertujuan meningkatkan kapasitas aparatur.'
  [sebelum CPT] selesai_rapi=True rasio_ulang=0.0 panjang=12
   This activity aims to enhance the capacity of government personnel.
----------------------------------------------------------------------------------------------------
INSTRUKSI: Apa ibu kota Provinsi Jawa Tengah? Jawab satu kata.
  [sebelum CPT] selesai_rapi=True rasio_ulang=0.0 panjang=4
   Semarang
----------------------------------------------------------------------------------------------------


Both `max_new_tokens` (=200) and `max_length`(=262144) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


INSTRUKSI: Tuliskan balasan e-mail sopan untuk menolak undangan rapat karena bentrok jadwal.
  [sebelum CPT] selesai_rapi=False rasio_ulang=0.04 panjang=200
   Tentu, berikut adalah contoh balasan email yang sopan dan profesional untuk menolak undangan rapat karena bentrok jadwal:  ---  **Subject:** Terima Kasih atas Undangan Rapat – Maaf Tidak Bisa Ikut  Halo [Nama Pengirim],  Terima kasih banyak atas undangan Anda dalam rapat [nama atau topik rapat]. Saya sangat menghargai kesempatan untuk berpartisipasi dan menyadari pentingnya pertemuan ini.  Sayangn
----------------------------------------------------------------------------------------------------
INSTRUKSI: Format jawaban sebagai daftar bernomor 1-3: tiga manfaat transformasi digital bagi pelayanan publik.
  [sebelum CPT] selesai_rapi=True rasio_ulang=0.007 panjang=138
   1. Meningkatkan aksesibilitas layanan bagi masyarakat melalui platform digital, seperti aplikasi atau website resmi, sehingga warga dapat mengakses layanan tan

4604

## 7. Pasang LoRA **baru** (fresh) di atas Instruct
**(INSTRUCT)** Berbeda dari notebook Base yang *melanjutkan* adapter fondasi: di sini kita pasang LoRA **baru**
karena adapter yang dilatih di atas bobot Base **tidak valid** dipindah ke bobot Instruct. Target modul & `r`
sama dengan run Base agar kapasitas adapter setara (perbandingan adil).

In [7]:
assert FRESH_LORA, 'Notebook ini dirancang untuk LoRA baru di atas Instruct (FRESH_LORA=True).'
model = FastLanguageModel.get_peft_model(
    model, r=LORA_R, lora_alpha=LORA_R, lora_dropout=0, bias='none',
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    use_rslora=True, use_gradient_checkpointing='unsloth', random_state=SEED)
model.print_trainable_parameters()

Unsloth 2026.11.3 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


Unsloth: compiling each transformer block as one region (Qwen3DecoderLayer).
trainable params: 33,030,144 || all params: 4,055,498,240 || trainable%: 0.8145


## 7b. Uji muat & estimasi waktu (sebelum training sungguhan)
Menjalankan beberapa blok nyata lewat forward+backward (bagian paling berat memori), mencatat **puncak VRAM**
dan **detik per blok**. Tidak ada `optimizer.step()` -> bobot LoRA tidak berubah. Angka VRAM paling bisa dipercaya;
estimasi waktu kasar (lihat progress bar training yang sebenarnya setelah ~20 langkah).

In [8]:
FastLanguageModel.for_training(model)
dtype = torch.bfloat16 if is_bfloat16_supported() else torch.float16
torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
times = []
try:
    for i in range(8):
        x = torch.from_numpy(train_blocks[i].astype(np.int64))[None].cuda()
        t0 = time.time()
        with torch.autocast('cuda', dtype=dtype):
            loss = model(input_ids=x, labels=x).loss
        loss.backward(); model.zero_grad(set_to_none=True); torch.cuda.synchronize()
        times.append(time.time() - t0)
        print(f'blok {i}: {times[-1]:5.1f} detik | puncak VRAM {torch.cuda.max_memory_allocated() / 2**30:.2f} GB')
except torch.cuda.OutOfMemoryError:
    torch.cuda.empty_cache()
    raise RuntimeError('OOM: Instruct 4B tidak muat. Tutup aplikasi GPU lain, atau ganti MODEL_SIZE ke varian lebih kecil.')
sec = float(np.median(times[3:]))
est_h = len(train_blocks) * sec * 1.05 / 3600
print(f'\n{TAG}: {sec:.1f} detik/blok = {MAX_SEQ_LEN / sec:.0f} token/detik | puncak VRAM {torch.cuda.max_memory_reserved() / 2**30:.2f} GB dari {total_b / 2**30:.2f} GB')
print(f'Estimasi {len(train_blocks) * MAX_SEQ_LEN / 1e6:.1f} jt token: +-{est_h:.1f} jam.')
print('Pembanding: Base mencapai komposisi data yang sama lewat 3 sesi terpisah, total +-10,9 jam '
      '(fondasi 4,79 jam + legal 5,21 jam + kominfo 0,85 jam). Run Instruct gabungan 1-pass ini di kisaran '
      'serupa (+-10-12 jam) -- realistis butuh 1-2 sesi Kaggle dengan resume checkpoint otomatis.')

Unsloth: Will smartly offload gradients to save VRAM!
blok 0:  63.2 detik | puncak VRAM 3.89 GB
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.
blok 1:   1.5 detik | puncak VRAM 3.92 GB
blok 2:   1.5 detik | puncak VRAM 3.92 GB
blok 3:   1.5 detik | puncak VRAM 3.92 GB
blok 4:   1.5 detik | puncak VRAM 3.92 GB
blok 5:   1.6 detik | puncak VRAM 3.92 GB
blok 6:   1.6 detik | puncak VRAM 3.92 GB
blok 7:   1.6 detik | puncak VRAM 3.92 GB

4b_instruct: 1.6 detik/blok = 658 token/detik | puncak VRAM 4.21 GB dari 14.56 GB
Estimasi 25.1 jt token: +-11.1 jam.
Pembanding: Base mencapai komposisi data yang sama lewat 3 sesi terpisah, total +-10,9 jam (fondasi 4,79 jam + legal 5,21 jam + kominfo 0,85 jam). Run Instruct gabungan 1-pass ini di kisaran serupa (+-10-12 jam) -- realistis butuh 1-2 sesi Kaggle dengan resume checkpoint otomatis.


## 8. Training
`Trainer` standar dari transformers (bukan `SFTTrainer`) karena data **sudah** ter-tokenisasi dan ter-packing.
Jadi tidak ada pemrosesan tersembunyi yang bisa mengubah data. Setiap contoh: `input_ids` = 1024 token,
`labels` = salinannya (model menebak token berikutnya di **semua** posisi, termasuk EOS).

- **LR cosine + warmup 3%**: LR naik pelan di awal, lalu turun halus.
- **`eval_loss` tiap 100 langkah**: kalau `train_loss` turun tapi `eval_loss` naik, model mulai menghafal.
- **Checkpoint tiap 25 langkah + resume otomatis**: kalau kernel mati/sesi habis, jalankan ulang semua sel
  dan training lanjut dari checkpoint terakhir. **Hapus folder `CKPT_DIR` jika hyperparameter diubah.**

**Catatan volume:** total ±25,1 juta token dalam 1 epoch diperkirakan ±1.532 langkah optimizer, sekitar
±10-12 jam (lihat sel 7b) - mendekati/melebihi batas sesi Kaggle 12 jam. `MAX_HOURS=11` akan menghentikan
training dengan rapi & menyimpan checkpoint sebelum batas sesi tercapai; jalankan ulang semua sel di sesi
baru untuk **resume otomatis** sampai `trainer.train()` selesai 1 epoch penuh.

In [9]:
from transformers import Trainer, TrainingArguments, default_data_collator

class Blocks(torch.utils.data.Dataset):
    def __init__(self, arr): self.arr = torch.from_numpy(np.ascontiguousarray(arr).astype(np.int64))
    def __len__(self): return len(self.arr)
    def __getitem__(self, i): x = self.arr[i]; return {'input_ids': x, 'labels': x.clone()}

train_ds = Blocks(train_blocks)
val_ds   = Blocks(to_blocks(val_ids))
total_steps = math.ceil(len(train_ds) / (BATCH * GRAD_ACC))
print(f'train {len(train_ds):,} blok | val {len(val_ds):,} blok | {total_steps} langkah optimizer')

args = TrainingArguments(
    output_dir=str(CKPT_DIR),
    per_device_train_batch_size=BATCH, gradient_accumulation_steps=GRAD_ACC, per_device_eval_batch_size=1,
    num_train_epochs=1, learning_rate=LR, lr_scheduler_type='cosine',
    warmup_steps=max(1, int(0.03 * total_steps)), weight_decay=0.01, optim='adamw_8bit',
    bf16=is_bfloat16_supported(), fp16=not is_bfloat16_supported(),
    logging_steps=10, eval_strategy='steps', eval_steps=EVAL_STEPS,
    save_strategy='steps', save_steps=SAVE_STEPS, save_total_limit=2,
    dataloader_num_workers=0, remove_unused_columns=False, report_to='none', seed=SEED)

from transformers import TrainerCallback
MAX_HOURS = 11.0

class TimeGuard(TrainerCallback):
    def __init__(self): self.t0 = time.time()
    def on_step_end(self, args, state, control, **kw):
        if time.time() - self.t0 > MAX_HOURS * 3600:
            print(f'Batas {MAX_HOURS} jam di langkah {state.global_step}: berhenti & simpan.')
            control.should_save = True; control.should_training_stop = True
        return control

trainer = Trainer(model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds,
                  data_collator=default_data_collator, processing_class=tokenizer, callbacks=[TimeGuard()])

FastLanguageModel.for_training(model)
ckpts = sorted(CKPT_DIR.glob('checkpoint-*'), key=lambda p: int(p.name.split('-')[1]))
print('Resume dari', ckpts[-1].name if ckpts else '- (mulai dari awal)')
trainer.train(resume_from_checkpoint=str(ckpts[-1]) if ckpts else None)

train 24,510 blok | val 329 blok | 1532 langkah optimizer
Resume dari - (mulai dari awal)


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
/usr/local/lib/python3.13/dist-packages/unsloth/models/_utils.py:487: UserWarning: Unsloth: detected a manual forward/backward run before trainer.train(); reset the torch.compile graph cache it poisoned so training starts clean. To avoid this, run any pre-train probe under `with torch.no_grad():`.
  warnings.warn(
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 24,510 | Num Epochs = 1 | Total steps = 1,532
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 16
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 16 x 1) = 16
 "-____-"     Trainable parameters = 33,030,144 of 4,055,498,240 (0.81% trained)


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss,Validation Loss
100,1.462790,1.469012
200,1.414716,1.420747
300,1.370705,1.398339
400,1.390701,1.386241
500,1.380613,1.375297
600,1.413522,1.368098
700,1.440761,1.361570
800,1.410003,1.355016
900,1.268150,1.350095
1000,1.342779,1.346966


Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/_cpt_instruct_output/checkpoints_4b_instruct_merged/checkpoint-25/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/_cpt_instruct_output/checkpoints_4b_instruct_merged/checkpoint-50/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/_cpt_instruct_output/checkpoints_4b_instruct_merged/checkpoint-75/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/_cpt_instruct_output/checkpoints_4b_instruct_merged/checkpoint-100/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/_cpt_instruct_output/checkpoints_4b_instruct_merged/checkpoint-125/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/_cpt_instruct_output/checkpoints_4b_instruct_merged/checkpoint-150/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /kaggle/wo

Batas 11.0 jam di langkah 1468: berhenti & simpan.


Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/_cpt_instruct_output/checkpoints_4b_instruct_merged/checkpoint-1468/tokenizer_config.json.


TrainOutput(global_step=1468, training_loss=1.3811133914812384, metrics={'train_runtime': 39610.6003, 'train_samples_per_second': 0.619, 'train_steps_per_second': 0.039, 'total_flos': 5.2911968486424576e+17, 'train_loss': 1.3811133914812384, 'epoch': 0.9583027335781313})

### Ringkasan kurva loss
`train_loss` turun lalu melandai = normal. `eval_loss` ikut turun = belajar pola, bukan menghafal.
`eval_loss` naik di akhir = tanda overfit (turunkan LR / pakai checkpoint lebih awal).

In [10]:
hist = trainer.state.log_history
(OUT_DIR / f'log_history_{TAG}_merged.json').write_text(json.dumps(hist, indent=1))
rows = defaultdict(dict)                 # log train & eval tersimpan sebagai entri terpisah -> gabung per langkah
for h in hist:
    for k in ('loss', 'eval_loss', 'learning_rate'):
        if k in h: rows[h['step']][k] = h[k]
fmt = lambda v, f: format(v, f) if v is not None else '-'
print(f'{"step":>6} {"train_loss":>11} {"eval_loss":>10} {"lr":>10}')
for step, r in sorted(rows.items()):
    if step % 50 == 0 or 'eval_loss' in r or step == trainer.state.global_step:
        print(f'{step:>6} {fmt(r.get("loss"), ".4f"):>11} {fmt(r.get("eval_loss"), ".4f"):>10} {fmt(r.get("learning_rate"), ".2e"):>10}')

  step  train_loss  eval_loss         lr
    50      1.5386          -   5.00e-05
   100      1.4628     1.4690   4.98e-05
   150      1.4712          -   4.94e-05
   200      1.4147     1.4207   4.87e-05
   250      1.4454          -   4.77e-05
   300      1.3707     1.3983   4.65e-05
   350      1.3217          -   4.51e-05
   400      1.3907     1.3862   4.34e-05
   450      1.3163          -   4.15e-05
   500      1.3806     1.3753   3.94e-05
   550      1.4629          -   3.72e-05
   600      1.4135     1.3681   3.48e-05
   650      1.3145          -   3.23e-05
   700      1.4408     1.3616   2.98e-05
   750      1.3416          -   2.71e-05
   800      1.4100     1.3550   2.45e-05
   850      1.4321          -   2.19e-05
   900      1.2681     1.3501   1.93e-05
   950      1.5057          -   1.67e-05
  1000      1.3428     1.3470   1.43e-05
  1050      1.3043          -   1.20e-05
  1100      1.2998     1.3454   9.79e-06
  1150      1.3435          -   7.79e-06
  1200      1.42

## 9. Simpan adapter
Hanya adapter LoRA (puluhan MB). Untuk memakainya: muat **Qwen3-4B-Instruct-2507** yang sama + adapter ini.

In [11]:
model.save_pretrained(str(ADAPTER_DIR)); tokenizer.save_pretrained(str(ADAPTER_DIR))
print('Adapter tersimpan di', ADAPTER_DIR)

Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/_cpt_instruct_output/adapter_cpt_4b_instruct_merged/tokenizer_config.json.


Adapter tersimpan di /kaggle/working/_cpt_instruct_output/adapter_cpt_4b_instruct_merged


## 10. Perplexity sesudah CPT: bandingkan dengan baseline
Kolom **delta** = perubahan PPL (% negatif = membaik). Harapan: kominfo/legal turun paling tajam (domain
paling terasa, walau proporsinya kecil di data gabungan), news/wiki/socmed juga turun (replay-nya besar).
`run_4b_instruct_merged.json` disimpan untuk dibandingkan dengan hasil akhir kumulatif Base round-3:
**kominfo 2,874->2,183 (-24,1%), legal 2,894->2,118 (-26,8%), news 5,760->5,048 (-12,4%), wiki 9,280->8,319
(-10,3%), socmed 18,545->17,283 (-6,8%)**.

In [12]:
after = {s: perplexity(model, test_ids[s], f'sesudah {s}') for s in SOURCES if s in test_ids}
print(f'{"sumber":8s} {"PPL base":>9} {"PPL CPT":>9} {"delta":>8}')
for s in after:
    b, a = baseline[s]['ppl'], after[s]['ppl']
    print(f'{s:8s} {b:9.3f} {a:9.3f} {(a - b) / b:+8.2%}')

(OUT_DIR / f'run_{TAG}_merged.json').write_text(json.dumps({
    'model': MODEL_NAME, 'fresh_lora': FRESH_LORA, 'train_tokens': int(train_blocks.size),
    'max_seq_len': MAX_SEQ_LEN, 'lora_r': LORA_R, 'lr': LR, 'batch': BATCH * GRAD_ACC,
    'steps': trainer.state.global_step, 'baseline': baseline, 'after': after, 'adapter': str(ADAPTER_DIR)}, indent=1))

sesudah kominfo:   0%|          | 0/144 [00:00<?, ?it/s]

sesudah legal:   0%|          | 0/245 [00:00<?, ?it/s]

sesudah news:   0%|          | 0/245 [00:00<?, ?it/s]

sesudah wiki:   0%|          | 0/243 [00:00<?, ?it/s]

sesudah socmed:   0%|          | 0/174 [00:00<?, ?it/s]

sumber    PPL base   PPL CPT    delta
kominfo      3.562     2.248  -36.89%
legal        3.434     2.058  -40.08%
news         7.357     5.074  -31.04%
wiki        11.951     8.497  -28.91%
socmed      23.365    17.662  -24.41%


1295

## 10b. (INSTRUCT) Instruction-following SESUDAH CPT - ukur 'forgetting'
Prompt **sama persis** dengan sel 6b, dijalankan dengan adapter **dinyalakan** (hasil CPT) dan **dimatikan**
(`disable_adapter` = Instruct asli, sanity check). Lalu ditampilkan berdampingan **sebelum vs sesudah**.

**Cara membaca (ini inti jawaban untuk Komdigi):**
- Jawaban *sesudah* tetap mengikuti instruksi, format, dan bahasa rapi -> CPT-di-Instruct **tidak** merusak SFT (untuk skala/replay ini).
- Jawaban *sesudah* mulai "melanjutkan teks" (mengabaikan instruksi), format rusak, `selesai_rapi=False`,
  atau `rasio_ulang` naik tajam -> **konfirmasi** kekhawatiran Komdigi: CPT teks-mentah mengikis instruction-following.

In [13]:
FastLanguageModel.for_inference(model)
instr_after = []
print('='*100)
for p, before in zip(INSTR_PROMPTS, instr_before):
    aft = chat_generate(model, p)
    with model.disable_adapter():
        asli = chat_generate(model, p)
    instr_after.append({'cpt': aft, 'instruct_asli': asli})
    print('INSTRUKSI:', p)
    print(f'  [SEBELUM CPT  ] rapi={before["selesai_rapi"]} ulang={before["rasio_ulang"]} len={before["panjang"]}')
    print('    ', before['text'][:350].replace('\n', ' '))
    print(f'  [SESUDAH CPT  ] rapi={aft["selesai_rapi"]} ulang={aft["rasio_ulang"]} len={aft["panjang"]}')
    print('    ', aft['text'][:350].replace('\n', ' '))
    print('-'*100)

# ringkasan sinyal agregat
import statistics as st
def agg(items, key='rasio_ulang'):
    return round(st.mean(x[key] for x in items), 3)
print('\nRINGKASAN SINYAL (rata-rata):')
print(f'  rasio_ulang  sebelum={agg(instr_before):.3f}  sesudah={agg([a["cpt"] for a in instr_after]):.3f}')
print(f'  selesai_rapi sebelum={sum(x["selesai_rapi"] for x in instr_before)}/{len(instr_before)}'
      f'  sesudah={sum(a["cpt"]["selesai_rapi"] for a in instr_after)}/{len(instr_after)}')
(OUT_DIR / 'instr_after.json').write_text(json.dumps(
    [{'prompt': p, 'before': b, **a} for p, b, a in zip(INSTR_PROMPTS, instr_before, instr_after)],
    indent=1, ensure_ascii=False), encoding='utf-8')

Both `max_new_tokens` (=200) and `max_length`(=262144) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Both `max_new_tokens` (=200) and `max_length`(=262144) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=200) and `max_length`(=262144) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


INSTRUKSI: Buatkan 3 poin Latar Belakang singkat untuk TOR kegiatan pelatihan literasi digital aparatur.
  [SEBELUM CPT  ] rapi=False ulang=0.015 len=200
     Tentu, berikut adalah 3 poin latar belakang singkat yang dapat digunakan dalam dokumen TOR (Tujuan dan Objek Rencana) kegiatan pelatihan literasi digital aparatur:  1. Dalam era digitalisasi, kemampuan aparatur pemerintah dalam mengelola teknologi informasi menjadi kunci utama dalam meningkatkan efisiensi, transparansi, dan kualitas layanan publik.
  [SESUDAH CPT  ] rapi=False ulang=0.045 len=200
     Tentu, berikut adalah 3 poin latar belakang singkat yang sesuai dan relevan untuk digunakan dalam dokumen TOR (Tanpa Dokumen) kegiatan pelatihan literasi digital aparatur:  1. Dalam era digitalisasi yang semakin pesat, kemampuan aparatur pemerintah dalam mengelola informasi, menggunakan teknologi informasi, serta berinteraksi secara efektif melalui
-------------------------------------------------------------------------------------

Both `max_new_tokens` (=200) and `max_length`(=262144) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=200) and `max_length`(=262144) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


INSTRUKSI: Jelaskan dalam 2 kalimat apa itu RAB dalam pengadaan pemerintah.
  [SEBELUM CPT  ] rapi=True ulang=0.02 len=102
     RAB (Rencana Anggaran Bangunan) adalah dokumen yang menyajikan perhitungan biaya pembangunan suatu proyek secara rinci, termasuk harga material, tenaga kerja, dan biaya lainnya. Dalam pengadaan pemerintah, RAB digunakan sebagai dasar penilaian kebutuhan anggaran dan untuk menjamin bahwa proyek dapat dijalankan sesuai dengan anggaran yang tersedia.
  [SESUDAH CPT  ] rapi=True ulang=0.032 len=96
     RAB (Rincian Anggaran Biaya) adalah dokumen yang menyajikan detail biaya pelaksanaan suatu pekerjaan atau konstruksi dalam pengadaan pemerintah, mencakup harga satuan kerja dan total biaya perhitungan. Dokumen ini digunakan sebagai dasar penetapan anggaran serta pemantauan pelaksanaan pengadaan secara transparan dan akuntabel.
----------------------------------------------------------------------------------------------------


Both `max_new_tokens` (=200) and `max_length`(=262144) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=200) and `max_length`(=262144) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


... (8 outputs left out)

INSTRUKSI: Tuliskan balasan e-mail sopan untuk menolak undangan rapat karena bentrok jadwal.
  [SEBELUM CPT  ] rapi=False ulang=0.04 len=200
     Tentu, berikut adalah contoh balasan email yang sopan dan profesional untuk menolak undangan rapat karena bentrok jadwal:  ---  **Subject:** Terima Kasih atas Undangan Rapat – Maaf Tidak Bisa Ikut  Halo [Nama Pengirim],  Terima kasih banyak atas undangan Anda dalam rapat [nama atau topik rapat]. Saya sangat menghargai kesempatan untuk berpartisipas
  [SESUDAH CPT  ] rapi=False ulang=0.02 len=200
     Berikut adalah contoh balasan e-mail yang sopan dan profesional untuk menolak undangan rapat karena bentrok jadwal:  ---  **Subject:** Pengundangan Rapat – Permintaan Maaf, Tidak Bisa Menghadiri  Assalamu'alaikum Warahmatullahi Wabarakatuh,  Terima kasih atas undangan rapat yang telah diterima dengan baik. Saya sangat menghargai kesempatan ini sert
---------------------------------------------------------------------------------------------------

Both `max_new_tokens` (=200) and `max_length`(=262144) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


INSTRUKSI: Format jawaban sebagai daftar bernomor 1-3: tiga manfaat transformasi digital bagi pelayanan publik.
  [SEBELUM CPT  ] rapi=True ulang=0.007 len=138
     1. Meningkatkan aksesibilitas layanan bagi masyarakat melalui platform digital, seperti aplikasi atau website resmi, sehingga warga dapat mengakses layanan tanpa harus datang langsung ke kantor.  2. Mempercepat proses pelayanan dengan otomatisasi dan sistem e-formulir, yang mengurangi waktu tunggu dan meminimalkan kesalahan manusia.  3. Membangun t
  [SESUDAH CPT  ] rapi=True ulang=0.0 len=89
     1. Meningkatkan kualitas layanan melalui sistem yang lebih cepat, akurat, dan transparan.   2. Memperluas aksesibilitas pelayanan kepada masyarakat di seluruh wilayah, termasuk daerah terpencil.   3. Mengurangi biaya operasional dengan menggantikan proses administratif manual dengan otomatisasi.
----------------------------------------------------------------------------------------------------

RINGKASAN SINYAL (rata-rata):
  ras

12377

## 11. Uji kualitatif domain (gaya chat): CPT vs Instruct asli
Prompt domain TOR/RAB lewat chat template; adapter dimatikan vs dinyalakan. Yang diperhatikan: apakah jawaban
CPT lebih tepat istilah (Permen Kominfo, Dasar Hukum, struktur TOR) **tanpa** kehilangan gaya menjawab instruksi.

In [14]:
FastLanguageModel.for_inference(model)
domain_prompts = [
    'Tuliskan bagian "Dasar Hukum" untuk TOR layanan di Kementerian Komunikasi dan Digital, sebutkan jenis peraturannya.',
    'Jelaskan komponen utama yang harus ada dalam sebuah RAB kegiatan pemerintah.',
    'Buat kerangka TOR (poin-poin bab) untuk kegiatan pengawasan isi siaran.',
]
for p in domain_prompts:
    with model.disable_adapter(): base_out = chat_generate(model, p)
    cpt_out = chat_generate(model, p)
    print('PROMPT :', p)
    print('[INSTRUCT asli]:', base_out['text'][:400].replace('\n', ' '))
    print('[CPT        ] :', cpt_out['text'][:400].replace('\n', ' '))
    print('-' * 100)

Both `max_new_tokens` (=200) and `max_length`(=262144) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=200) and `max_length`(=262144) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=200) and `max_length`(=262144) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


PROMPT : Tuliskan bagian "Dasar Hukum" untuk TOR layanan di Kementerian Komunikasi dan Digital, sebutkan jenis peraturannya.
[INSTRUCT asli]: Berikut adalah contoh bagian **"Dasar Hukum"** untuk **Tujuan Operasional Rencana (TOR)** layanan di **Kementerian Komunikasi dan Digital Republik Indonesia**, disusun sesuai dengan jenis peraturan yang berlaku:  ---  ### **Dasar Hukum**  Penyelenggaraan layanan pada Kementerian Komunikasi dan Digital dilaksanakan berdasarkan berbagai peraturan perundang-undangan nasional yang telah ditetapkan ole
[CPT        ] : **Bagian: Dasar Hukum**  - a. Undang-Undang Nomor 1 Tahun 2004 tentang Perbendaharaan Negara (Lembaran Negara Republik Indonesia Tahun 2004 Nomor 5, Tambahan Lembaran Negara Republik Indonesia Nomor 4355); - b. Undang-Undang Nomor 15 Tahun 2004 tentang Pemeriksaan Pengelolaan dan Tanggung Jawab Keuangan Negara (Lembaran Negara Republik Indonesia Tahun 2004 Nomor 66, Tambahan Lembaran Negara Republ
----------------------------------------

Both `max_new_tokens` (=200) and `max_length`(=262144) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=200) and `max_length`(=262144) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


PROMPT : Jelaskan komponen utama yang harus ada dalam sebuah RAB kegiatan pemerintah.
[INSTRUCT asli]: RAB (Rencana Anggaran Biaya) kegiatan pemerintah adalah dokumen penting yang digunakan untuk menentukan biaya pelaksanaan suatu proyek atau kegiatan pemerintahan secara sistematis, transparan, dan akuntabel. Komponen utama dari RAB kegiatan pemerintah harus mencakup elemen-elemen yang memungkinkan pengelolaan anggaran dengan efisien, sesuai peraturan perundang-undangan, serta dapat dipantau oleh b
[CPT        ] : Rencana Anggaran Biaya (RAB) merupakan dokumen penting dalam pelaksanaan kegiatan pemerintahan, terutama pada proyek atau kegiatan yang memerlukan alokasi anggaran. Dalam konteks pemerintah, RAB kegiatan harus lengkap dan transparan agar sesuai dengan ketentuan peraturan perundang-undangan serta prinsip akuntabilitas publik.  Berikut adalah **komponen utama** yang harus ada dalam sebuah RAB kegiat
-------------------------------------------------------------------------------

Both `max_new_tokens` (=200) and `max_length`(=262144) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


PROMPT : Buat kerangka TOR (poin-poin bab) untuk kegiatan pengawasan isi siaran.
[INSTRUCT asli]: Berikut adalah **kerangka TOR (Tujuan, Objek, Ruang Lingkup)** untuk kegiatan **pengawasan isi siaran**, disusun dalam bentuk poin-poin bab yang jelas dan terstruktur:  ---  ### **Kerangka TOR – Pengawasan Isi Siaran**  #### **1. Tujuan** - Menjamin kualitas, akurasi, dan kepatuhan isi siaran terhadap peraturan hukum, etika, nilai-nilai nasional, serta standar komunikasi publik. - Mencegah penyeba
[CPT        ] : Berikut adalah contoh **Kerangka Tor (Tujuan Operasional Ringkas)** untuk kegiatan **Pengawasan Isi Siaran** yang dapat digunakan sebagai panduan dalam penyusunan dokumen perencanaan kegiatan, seperti RKA/DIPA atau Laporan Kegiatan:  ---  ### **KERANGKA TUJUAN OPERASIONAL RINGKAS (TOR) – PENGAWASAN ISI SIARAN**  | BAB | POKOK BAHASA | TUJUAN OPERASIONAL | |------|------------------|---------------
------------------------------------------------------------------------------------

## 12. Bungkus hasil untuk diunduh
Checkpoint besar dihapus; yang dibungkus: adapter, log, `run_4b_instruct_*.json`, `instr_before.json`, `instr_after.json`.

In [15]:
shutil.rmtree(CKPT_DIR, ignore_errors=True)
zip_path = shutil.make_archive('/kaggle/working/hasil_cpt_instruct', 'zip', OUT_DIR)
print(zip_path, f'{os.path.getsize(zip_path) / 2**20:.0f} MB')

/kaggle/working/hasil_cpt_instruct.zip 148 MB


## Cara membandingkan dengan run Base (kumulatif 3 ronde) & langkah berikutnya

**Pembanding Base (hasil akhir kumulatif round-3, test set identik - lihat
`tim-A/cpt/kominfo/cpt-quen3-4b-id-kominfo-results.ipynb`):**

| sumber | Base kumulatif 3 ronde (delta PPL) |
|---|---|
| kominfo | 2.874 -> 2.183 (**-24,1%**) |
| legal | 2.894 -> 2.118 (**-26,8%**) |
| news | 5.760 -> 5.048 (-12,4%) |
| wiki | 9.280 -> 8.319 (-10,3%) |
| socmed | 18.545 -> 17.283 (-6,8%) |

**Catatan metodologi (baca sebelum menyimpulkan):** Base mencapai angka di atas lewat **3 sesi training
berurutan** (adapter dilanjutkan tiap ronde, total ±10,9 jam: fondasi 4,79 jam + legal 5,21 jam + kominfo
0,85 jam). Instruct di notebook ini memakai **1 pass gabungan** dari ketiga cache yang sama (LoRA baru,
±10-12 jam). Datanya identik secara volume & sumber, tapi **prosedurnya berbeda** (sekuensial vs gabungan
sekali jalan) - jadi perbandingan ini adalah A/B yang wajar untuk pertanyaan "Base vs Instruct", bukan
replikasi prosedur yang 100% identik.

**Interpretasi hasil Instruct vs Base:**
1. **Delta PPL sebanding** (terutama kominfo/legal turun tajam) **dan** instruction-following **bertahan**
   (sel 10b) -> CPT-di-Instruct (bahkan dengan volume data domain sebesar ini) aman sebagai jalan pintas.
   Laporkan ke Komdigi sebagai bukti tandingan terkontrol.
2. **Instruction-following turun** (format rusak / "melanjutkan teks" / `selesai_rapi` anjlok) ->
   konfirmasi masukan Komdigi; tetap pakai jalur **Base -> CPT -> SFT**.
3. **Delta PPL domain jauh lebih kecil dari Base** -> kemungkinan karena 1-pass gabungan kurang efektif
   dibanding sekuensial (kominfo/legal "tenggelam" di antara replay besar), bukan berarti CPT-di-Instruct
   gagal total - strategi *curriculum* (sekuensial, seperti Base) mungkin perlu direplikasi juga di Instruct
   untuk perbandingan yang lebih adil.

**Langkah berikutnya (poin 4):** JANGAN eksekusi sebelum tim menjawab validitas pendekatan. Begitu disetujui,
jalankan notebook ini apa adanya (A/B 1 ronde gabungan), lalu tempelkan tabel delta + ringkasan sel 10b ke
laporan. Kalau kontrol ini aman tapi hasil domain kurang tajam karena efek 1-pass vs sekuensial, ronde
lanjutan bisa coba replikasi 3 sesi berurutan di Instruct juga (LoRA yang sama dilanjutkan antar ronde) untuk
pembanding yang lebih presisi.